# Textile Pattern GAN — Exploratory Data Analysis

**Scope:** Batik datasets (`Batik_Lasem`, `Batik_Nitik_960`, `Batik_Nitik_Sarimbit_120`,
`Original Images`) and the **NeuralLoom** dataset.
**Excluded on purpose:** `DeepFashion2` and `DeepFashion3D`.

This notebook mounts the shared Google Drive folder and runs a full pixel-level EDA
over every included class. It is the reproducible companion to `eda/generate_eda.py`
(which runs on a local sample) and `eda/EDA_REPORT.md` (written findings).

Run order: (1) mount Drive, (2) set `ROOT`, (3) run all cells.

## 1. Setup

In [ ]:
# In Colab:
from google.colab import drive
drive.mount('/content/drive')
# Then copy the shared folder into *your* Drive ("Add shortcut to Drive"),
# or download it locally. Set ROOT to the folder that contains the dataset dirs.


In [ ]:
import os, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings('ignore')

# EDIT THIS: path to the folder that holds Batik_Lasem / Batik_Nitik_960 / NeuralLoom / ...
ROOT = '/content/drive/MyDrive/Datasets'

EXCLUDE = ('DeepFashion2', 'DeepFashion3D')      # never analysed
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
print('ROOT exists:', os.path.isdir(ROOT))
print('top-level:', [d for d in os.listdir(ROOT)] if os.path.isdir(ROOT) else 'N/A')


## 2. Class discovery & composition

We treat every leaf directory (a folder that directly contains images) as a
"class bucket" and label it by its dataset + path, skipping anything under the
excluded DeepFashion trees.

In [ ]:
def is_excluded(path):
    parts = os.path.relpath(path, ROOT).split(os.sep)
    return any(p in EXCLUDE for p in parts)

records = []   # one row per image
for dirpath, _dirs, files in os.walk(ROOT):
    if is_excluded(dirpath):
        continue
    imgs = [f for f in files if f.lower().endswith(IMG_EXT)]
    if not imgs:
        continue
    rel = os.path.relpath(dirpath, ROOT)
    dataset = rel.split(os.sep)[0]
    for f in imgs:
        records.append(dict(dataset=dataset, folder=rel, path=os.path.join(dirpath, f)))

df = pd.DataFrame(records)
print(f'Total images (excluding DeepFashion): {len(df):,}')
comp = df.groupby('dataset').size().sort_values(ascending=False)
display(comp.to_frame('images'))

comp.plot(kind='bar', figsize=(9,4), color='#c0392b', title='Images per dataset (DeepFashion excluded)')
plt.ylabel('images'); plt.tight_layout(); plt.show()


In [ ]:
# Per-folder (finer granularity, e.g. individual Batik_Lasem motif classes)
per_folder = df.groupby('folder').size().sort_values(ascending=False)
display(per_folder.head(40).to_frame('images'))


## 3. Pixel-level statistics

For speed on large classes we sample up to `MAX_PER_FOLDER` images per folder.
Set it to `None` to process everything.

In [ ]:
MAX_PER_FOLDER = 300      # set None for the full dataset (slow)

def sample_paths(df, k):
    if k is None:
        return df
    return df.groupby('folder', group_keys=False).apply(
        lambda g: g.sample(min(len(g), k), random_state=0))

sdf = sample_paths(df, MAX_PER_FOLDER).reset_index(drop=True)

rows = []
for _, r in sdf.iterrows():
    try:
        im = Image.open(r.path); im.load()
    except Exception:
        rows.append(dict(path=r.path, dataset=r.dataset, folder=r.folder, corrupt=True)); continue
    a = np.asarray(im.convert('RGB')).astype(np.float32)
    w, h = im.size
    meta = sum(k in im.info for k in ('exif','photoshop','xmp','icc_profile'))
    rows.append(dict(path=r.path, dataset=r.dataset, folder=r.folder, corrupt=False,
                     width=w, height=h, pixels=w*h,
                     filesize_kb=os.path.getsize(r.path)/1024,
                     mean_r=a[...,0].mean(), mean_g=a[...,1].mean(), mean_b=a[...,2].mean(),
                     brightness=a.mean(), contrast_std=a.std(), metadata_blocks=meta))
stats = pd.DataFrame(rows)
good = stats[~stats.corrupt]
print(f'Analysed {len(good):,} images; corrupt: {int(stats.corrupt.sum())}')
display(good.groupby('dataset')[['width','height','brightness','contrast_std','filesize_kb']].mean().round(1))


### 3a. Image dimensions

In [ ]:
dim = good.groupby(['width','height']).size().sort_values(ascending=False)
display(dim.head(15).to_frame('count'))
plt.figure(figsize=(6,5)); plt.scatter(good.width, good.height, alpha=.2)
plt.xlabel('width'); plt.ylabel('height'); plt.title('Image dimensions'); plt.show()


### 3b. Metadata bloat (bytes per pixel)

In [ ]:
good = good.assign(bytes_per_px=good.filesize_kb*1024/good.pixels)
display(good.groupby('dataset').bytes_per_px.mean().round(1).to_frame('bytes/px (raw RGB=3)'))
plt.figure(figsize=(6,4)); plt.scatter(good.pixels, good.filesize_kb, alpha=.3)
plt.xlabel('pixels'); plt.ylabel('KB'); plt.title('File size vs pixels'); plt.show()


### 3c. Brightness & colour

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
for d,g in good.groupby('dataset'):
    ax[0].hist(g.brightness, bins=30, alpha=.5, label=d)
ax[0].set_title('Brightness by dataset'); ax[0].set_xlabel('0-255'); ax[0].legend(fontsize=7)
for ch,c in [('mean_r','r'),('mean_g','g'),('mean_b','b')]:
    ax[1].hist(good[ch], bins=30, alpha=.5, color=c, label=ch)
ax[1].set_title('RGB channel means'); ax[1].legend()
plt.tight_layout(); plt.show()


### 3d. Sample grids per dataset

In [ ]:
for d, g in good.groupby('dataset'):
    paths = g.path.sample(min(16, len(g)), random_state=1).tolist()
    cols = 8; rows_ = int(np.ceil(len(paths)/cols))
    plt.figure(figsize=(cols*1.4, rows_*1.4))
    for i,p in enumerate(paths):
        ax=plt.subplot(rows_, cols, i+1); ax.imshow(Image.open(p).convert('RGB')); ax.axis('off')
    plt.suptitle(d); plt.tight_layout(); plt.show()


## 4. Data-quality checks
Duplicate detection (exact + near-duplicate via average hash) and corrupt files.

In [ ]:
import hashlib
def ahash(p, s=8):
    im = Image.open(p).convert('L').resize((s,s))
    a = np.asarray(im); return (a > a.mean()).flatten()
hashes = {}
for p in good.path:
    try:
        h = ''.join('1' if b else '0' for b in ahash(p)); hashes.setdefault(h, []).append(p)
    except Exception: pass
dups = {h:v for h,v in hashes.items() if len(v) > 1}
print(f'near-duplicate groups (aHash): {len(dups)}')
print(f'corrupt/unreadable files: {int(stats.corrupt.sum())}')


## 5. GAN-readiness notes
See `eda/EDA_REPORT.md` for the full write-up. Key modelling implications:
- Class imbalance across Batik_Lasem motifs (Latohan/Seritan/Nyuk Pitu ~1.3-1.6x Gunung Ringgit).
- Mixed native resolutions (28x28 vs 142x142 vs originals) — pick one target size.
- Strip Photoshop/EXIF metadata before training (heavy bloat, no signal).
- NeuralLoom images are full photographs (very different domain from 28x28 batik crops);
  train separate models or condition on dataset.